# Part 2: Logistic Regression from Scratch

Scenario: After performing EDA and clustering analysis on the Maine legislative bills dataset, you want to test whether the bill's assigned committee can be predicted based on the title and text embeddings. You will implement logistic regression from scratch to perform this classification task. To keep things simple, we have picked just one category to predict (i.e., a binary classification problem). I have provided you with the boolean labels for whether a bill was assigned to the "Housing and Economic Development" committee in the `y.json` file.

In this notebook, you will implement logistic regression from scratch using only NumPy, train it with gradient descent, and compare its performance when using `text_embedding` vs. `title_embedding` as features.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
# Load the dataset from data/X.json and data/y.json.
df_X = pd.read_json('data/X.json')
y = pd.read_json('data/y.json')['committee_bool'].values

X_title = np.stack(df_X['title_embedding'].values)
X_text = np.stack(df_X['text_embedding'].values)


In [3]:
# Create train/test splits with random seed 6140, 80/20 split, stratified by y
X_title_train, X_title_test, y_train, y_test = train_test_split(
    X_title, y, test_size=0.2, random_state=6140, stratify=y
)
X_text_train, X_text_test, _, _ = train_test_split(
    X_text, y, test_size=0.2, random_state=6140, stratify=y
)

# Standardize the features using StandardScaler.
scaler_title = StandardScaler()
X_title_train = scaler_title.fit_transform(X_title_train)
X_title_test = scaler_title.transform(X_title_test)

scaler_text = StandardScaler()
X_text_train = scaler_text.fit_transform(X_text_train)
X_text_test = scaler_text.transform(X_text_test)

print(f"Training set: {X_title_train.shape[0]} samples")
print(f"Test set: {X_title_test.shape[0]} samples")
print(f"Positive class in train: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Positive class in test: {y_test.sum()} ({y_test.mean():.1%})")


Training set: 1043 samples
Test set: 261 samples
Positive class in train: 81 (7.8%)
Positive class in test: 20 (7.7%)


In [4]:
class LogisticRegression:
    def __init__(self, learning_rate=0.01, num_iterations=1000):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.theta = None  # Parameters to be learned (includes bias as first element)

    def sigmoid(self, z):
        # Sigmoid function, clipping z to prevent exp overflow
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _add_intercept(self, X):
        # Prepend a column of ones to X for the bias/intercept term
        return np.column_stack([np.ones(X.shape[0]), X])

    def fit(self, X, y):
        X = self._add_intercept(X)
        self.theta = np.zeros(X.shape[1])  # theta[0] is the bias, theta[1:] are the weights
        m = len(y)
        
        for _ in range(self.num_iterations):
            z = np.dot(X, self.theta)
            h = self.sigmoid(z)
            gradient = np.dot(X.T, (h - y)) / m
            self.theta -= self.learning_rate * gradient

    def predict(self, X):
        X = self._add_intercept(X)
        z = np.dot(X, self.theta)
        predicted_probs = self.sigmoid(z)
        return (predicted_probs >= 0.5).astype(int)


In [5]:
# Train two logistic regression models
# Note: I tested different learning rates. With standardized data, lr=0.5 and 1500 iterations
# gave me stable convergence.
model_text = LogisticRegression(learning_rate=0.5, num_iterations=1500)
model_text.fit(X_text_train, y_train)

model_title = LogisticRegression(learning_rate=0.5, num_iterations=1500)
model_title.fit(X_title_train, y_train)


In [6]:
# Predict class labels for each test set
preds_text = model_text.predict(X_text_test)
preds_title = model_title.predict(X_title_test)

print("========== Title Embeddings ==========")
print(f"Accuracy:  {accuracy_score(y_test, preds_title):.4f}")
print(f"Precision: {precision_score(y_test, preds_title, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, preds_title, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_test, preds_title, zero_division=0):.4f}")

print("\n========== Text Embeddings ==========")
print(f"Accuracy:  {accuracy_score(y_test, preds_text):.4f}")
print(f"Precision: {precision_score(y_test, preds_text, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, preds_text, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_test, preds_text, zero_division=0):.4f}")


========== Title Embeddings ==========
Accuracy:  0.9157
Precision: 0.4545
Recall:    0.5000
F1 Score:  0.4762

========== Text Embeddings ==========
Accuracy:  0.9464
Precision: 0.6250
Recall:    0.7500
F1 Score:  0.6818




**1. Which feature — `text_embedding` or `title_embedding` — produced better classification performance? Why do you think that is the case?**

The text_embedding outperformed the title_embedding by a significant margin for me (jumping from a 0.47 F1 score to 0.68, and recall went from 50% to 75%). This makes a lot of intuitive sense when I look at the data. Legislative bill titles are often confusingly brief or use opaque jargon like "An Act To Amend The Statutes" that does not actually tell you what the bill does. By using the full text embeddings, the model gets to see the actual domain-specific vocabulary and semantic context of the bill, giving it a much more robust signal to figure out if it is related to housing or the economy.

**2. Suppose this classifier is being used to flag potential Housing bills for a human reviewer. Which error metric (accuracy, precision, recall, or F1-score) is most important in this scenario? Which is least important? Justify your answer.**

For a "human-in-the-loop" review system like this, I firmly believe Recall is the most importat metric. Our main goal here is to ensure the reviewer does not miss any relevant Housing bills, so minimizing False Negatives is the key. It is totally fine if the human reviewer has to toss out a few False Positives, but missing a crucial bill completely defeats the purpose of the flagging tool! 
Conversely, Accuracy is the least useful metric here because our dataset is wildly imbalanced. A simple  normal model could just predict '0' for every single bill and still achieve over 92% accuracy, which looks great on paper but its kind of no use for the reviewer.

**3. How does the choice of learning rate affect the convergence of gradient descent? What strategies can be used to choose an appropriate learning rate?**

From my testing, the learning rate essentially shows our "step size" toward the minimum cost. When I tried making it extremely small, the model took forever to converge and required thousands of unnecessary iterations. But when its too large, it overshoots the minimum entirely and the cost function bounces around without settling. A strategy that worked well for me was starting with a small baseline value (like 0.01) and scaling it up experimentally on a validation set until the loss started behaving erratically. In a real-world scenario outside of this scratch implementation, I'd definitely want to use an adaptive learning rate optimizer like Adam, which automatically shrinks the step size as it gets closer to the minimum to safely lock in